# P07 SkyOps — Week 4: Source-to-Bronze Ingestion

### ZENAIZ × BVRIT Hyderabad Data Engineering Internship

**Project:** SkyOps Airline Delay Command Center  
**Project ID:** P07  
**Week:** 4 — Source-to-Bronze Ingestion  
**Primary implementation language:** Spark SQL  
**Databricks Volume:** `/Volumes/p07-skyops/default/skyops`

This notebook is the **project-specific Week 4 implementation** derived from the supplied SkyOps project pack, Week 3 notebook, and the Week 4 learning material.

### Week 4 goal

Move the four approved batch source files into persistent Bronze Delta tables while preserving the source business values and adding ingestion/lineage metadata.

```text
Approved CSV files
        ↓
Temporary source views
        ↓
Bronze-ready views
        ↓
Persistent Bronze Delta tables
        ↓
Reconciliation + metadata validation
        ↓
Controlled repeat-run + Delta history
```

**Important boundary:** Week 4 does **not** perform Silver cleaning, type correction, deduplication, business-rule validation, quarantine, Gold aggregation, Power BI, or streaming.

## How to use this notebook

Run the notebook **top to bottom, one executable cell at a time** on Databricks.

For each code cell:

1. Read the explanation immediately above it.
2. Verify the path/table name.
3. Run the cell.
4. Inspect the result.
5. Complete the checkpoint before continuing.

Do not replace the actual execution results with values typed into markdown. Your evidence must come from your Databricks run.

## Week 4 objectives

By the end of this notebook you should be able to:

1. Explain why the Bronze layer is needed.
2. Identify the four approved SkyOps batch sources.
3. Read the CSV sources from the Unity Catalog Volume.
4. Preserve source business columns in Bronze.
5. Add source, ingestion, schema-version and record-hash metadata.
6. Create one persistent Delta Bronze table per approved source.
7. Reconcile source and Bronze row counts.
8. Verify required metadata is populated.
9. Perform a controlled repeat-run without unintended row multiplication.
10. Inspect Delta history and explain the Source → Bronze flow.

## Week 4 source inventory

The supplied SkyOps source manifest defines four approved batch files:

| Source | Format | Grain | Business key / identifier | Bronze target |
|---|---|---|---|---|
| `airports.csv` | CSV | one approved airport code | `airport_code` | `bronze_airports` |
| `carriers.csv` | CSV | one reporting carrier code | `carrier_code` | `bronze_carriers` |
| `routes.csv` | CSV | one directed origin-destination pair | `route_id` / origin+destination | `bronze_routes` |
| `flights.csv` | CSV | one scheduled flight occurrence / physical source record | `source_record_key` | `bronze_flights` |

The project pack reports the controlled source sizes as 20 airports, 10 carriers, 360 routes and 123,337 flight records. These are **reference expectations only**; this notebook obtains the actual counts from Databricks.

## Bronze contract used here

Bronze is the first persistent Delta copy of an approved source.

### Preserve

- source business column names
- source business values
- source file boundaries
- source identifiers
- source traceability fields already present in the source

### Add technical metadata

- `_source_file_name`
- `_source_file_path`
- `_ingested_at`
- `_ingestion_run_id`
- `_schema_version`
- `_record_hash`
- `_rescued_payload` where parser context is available
- `_source_row_number` for reference sources where the source itself does not provide one

### Do not do in Week 4

- no business-value cleaning
- no type casting for Silver semantics
- no deduplication
- no HHMM validation
- no route/carrier/airport DQ enforcement
- no quarantine
- no Silver/Gold tables

## Important correction from the earlier Week 4 notebook

This version keeps the SkyOps source contract clearer:

- `flights.csv` already contains `source_record_key`, `source_period`, `source_original_filename` and `source_row_number`; those source fields are **preserved**, not replaced by generated values.
- For `airports`, `carriers` and `routes`, `_source_row_number` is generated deterministically from the source business key because those files do not contain an original row-number field.
- `_source_period` is **not fabricated** for reference files. It is not part of their approved business schema.
- Numeric-looking Bronze fields remain `STRING` when explicitly reading the raw CSVs; interpretation belongs to later Silver work.

# Part 1 — Prepare Databricks

Before running:

- Attach the notebook to Databricks compute.
- Make sure the four CSV files are present in `/Volumes/p07-skyops/default/skyops`.
- Keep the source filenames unchanged.
- Use the approved catalog/schema for the Bronze tables.

In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT
  current_catalog() AS active_catalog,
  current_schema() AS active_schema;

### Checkpoint

The result should show the catalog and schema where you intend to create the Bronze tables.

If your approved workspace uses a different catalog/schema, change only those two context statements after confirming the approved location.

In [ ]:
%fs
ls /Volumes/p07-skyops/default/skyops

### Checkpoint

The listing must contain:

```text
airports.csv
carriers.csv
routes.csv
flights.csv
```

Do not continue if a required file is missing or the filename is different.

# Part 2 — Controlled Week 4 run

A controlled run ID lets us identify which ingestion run produced the Bronze rows.

The same run ID is kept during the repeat-run test. A new run ID should only be used for a genuinely new approved batch.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW week4_run_control AS
SELECT
  'W04_SKYOPS_RUN01' AS ingestion_run_id,
  'skyops_source_v1.0' AS schema_version;

In [ ]:
%sql
SELECT *
FROM week4_run_control;

### Checkpoint

Confirm that the run ID and schema version are visible before building any Bronze table.

# Part 3 — Build `bronze_airports`

`airports.csv` is a small reference source.

For Bronze ingestion, every source field is read as `STRING`. This deliberately avoids applying Silver-level business typing or transformations at the landing stage.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW airports_source
(
  airport_code STRING,
  airport_name STRING,
  city STRING,
  state_region STRING,
  active_flag STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/p07-skyops/default/skyops/airports.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);

In [ ]:
%sql
SELECT *
FROM airports_source
LIMIT 10;

In [ ]:
%sql
DESCRIBE airports_source;

In [ ]:
%sql
SELECT COUNT(*) AS airports_source_count
FROM airports_source;

### Checkpoint — airports source

Verify:

- all five approved business fields are present;
- values are readable;
- the source row count is recorded from the query;
- no filtering or cleaning has been applied.

## Add airports Bronze metadata

For airports, `_source_row_number` is generated using the approved business key ordering. This is an ingestion trace ordinal, not a claim about an original row number that existed in the CSV.

The record hash is built from the ordered business columns.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW airports_bronze_ready AS
SELECT
  s.airport_code,
  s.airport_name,
  s.city,
  s.state_region,
  s.active_flag,
  ROW_NUMBER() OVER (ORDER BY s.airport_code) AS _source_row_number,
  'airports.csv' AS _source_file_name,
  '/Volumes/p07-skyops/default/skyops/airports.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version,
  sha2(
    concat_ws(
      '||',
      coalesce(s.airport_code, '<NULL>'),
      coalesce(s.airport_name, '<NULL>'),
      coalesce(s.city, '<NULL>'),
      coalesce(s.state_region, '<NULL>'),
      coalesce(s.active_flag, '<NULL>')
    ),
    256
  ) AS _record_hash,
  s._corrupt_record AS _rescued_payload
FROM airports_source s
CROSS JOIN week4_run_control r;

In [ ]:
%sql
SELECT *
FROM airports_bronze_ready
LIMIT 10;

### Checkpoint

The Bronze-ready result should contain the original five business columns plus the technical `_` columns.

No business column should be renamed, filtered or cleaned.

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_airports
USING DELTA
AS
SELECT *
FROM airports_bronze_ready;

In [ ]:
%sql
DESCRIBE TABLE bronze_airports;

In [ ]:
%sql
DESCRIBE DETAIL bronze_airports;

In [ ]:
%sql
SELECT *
FROM bronze_airports
LIMIT 10;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM airports_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_airports) AS bronze_count,
  (SELECT COUNT(*) FROM bronze_airports)
    - (SELECT COUNT(*) FROM airports_source) AS count_difference,
  CASE
    WHEN (SELECT COUNT(*) FROM airports_source)
       = (SELECT COUNT(*) FROM bronze_airports)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

### Airports checkpoint

Proceed only when the source-to-Bronze reconciliation is `PASS`.

If it is not, inspect the source view, Bronze-ready view, table definition and write operation before continuing.

# Part 4 — Build `bronze_carriers`

The carriers source contains:

- `carrier_code`
- `carrier_name`
- `active_flag`

The same Bronze pattern is reused: read → inspect → add metadata → persist → reconcile.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW carriers_source
(
  carrier_code STRING,
  carrier_name STRING,
  active_flag STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/p07-skyops/default/skyops/carriers.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);

In [ ]:
%sql
SELECT *
FROM carriers_source
LIMIT 10;

In [ ]:
%sql
DESCRIBE carriers_source;

In [ ]:
%sql
SELECT COUNT(*) AS carriers_source_count
FROM carriers_source;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW carriers_bronze_ready AS
SELECT
  s.carrier_code,
  s.carrier_name,
  s.active_flag,
  ROW_NUMBER() OVER (ORDER BY s.carrier_code) AS _source_row_number,
  'carriers.csv' AS _source_file_name,
  '/Volumes/p07-skyops/default/skyops/carriers.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version,
  sha2(
    concat_ws(
      '||',
      coalesce(s.carrier_code, '<NULL>'),
      coalesce(s.carrier_name, '<NULL>'),
      coalesce(s.active_flag, '<NULL>')
    ),
    256
  ) AS _record_hash,
  s._corrupt_record AS _rescued_payload
FROM carriers_source s
CROSS JOIN week4_run_control r;

In [ ]:
%sql
SELECT *
FROM carriers_bronze_ready
LIMIT 10;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_carriers
USING DELTA
AS
SELECT *
FROM carriers_bronze_ready;

In [ ]:
%sql
DESCRIBE TABLE bronze_carriers;

In [ ]:
%sql
SELECT *
FROM bronze_carriers
LIMIT 10;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM carriers_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_carriers) AS bronze_count,
  (SELECT COUNT(*) FROM bronze_carriers)
    - (SELECT COUNT(*) FROM carriers_source) AS count_difference,
  CASE
    WHEN (SELECT COUNT(*) FROM carriers_source)
       = (SELECT COUNT(*) FROM bronze_carriers)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

### Carriers checkpoint

The reconciliation must show `PASS`.

# Part 5 — Build `bronze_routes`

The routes source contains six business fields.

`distance_miles` is intentionally read as `STRING` in Bronze. The later Silver layer can cast it to an appropriate numeric type after the Bronze copy has been validated.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW routes_source
(
  route_id STRING,
  origin_airport_code STRING,
  destination_airport_code STRING,
  route_label STRING,
  distance_miles STRING,
  distance_band STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/p07-skyops/default/skyops/routes.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);

In [ ]:
%sql
SELECT *
FROM routes_source
LIMIT 10;

In [ ]:
%sql
DESCRIBE routes_source;

In [ ]:
%sql
SELECT COUNT(*) AS routes_source_count
FROM routes_source;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW routes_bronze_ready AS
SELECT
  s.route_id,
  s.origin_airport_code,
  s.destination_airport_code,
  s.route_label,
  s.distance_miles,
  s.distance_band,
  ROW_NUMBER() OVER (ORDER BY s.route_id) AS _source_row_number,
  'routes.csv' AS _source_file_name,
  '/Volumes/p07-skyops/default/skyops/routes.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version,
  sha2(
    concat_ws(
      '||',
      coalesce(s.route_id, '<NULL>'),
      coalesce(s.origin_airport_code, '<NULL>'),
      coalesce(s.destination_airport_code, '<NULL>'),
      coalesce(s.route_label, '<NULL>'),
      coalesce(s.distance_miles, '<NULL>'),
      coalesce(s.distance_band, '<NULL>')
    ),
    256
  ) AS _record_hash,
  s._corrupt_record AS _rescued_payload
FROM routes_source s
CROSS JOIN week4_run_control r;

In [ ]:
%sql
SELECT *
FROM routes_bronze_ready
LIMIT 10;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_routes
USING DELTA
AS
SELECT *
FROM routes_bronze_ready;

In [ ]:
%sql
DESCRIBE TABLE bronze_routes;

In [ ]:
%sql
SELECT *
FROM bronze_routes
LIMIT 10;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM routes_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_routes) AS bronze_count,
  (SELECT COUNT(*) FROM bronze_routes)
    - (SELECT COUNT(*) FROM routes_source) AS count_difference,
  CASE
    WHEN (SELECT COUNT(*) FROM routes_source)
       = (SELECT COUNT(*) FROM bronze_routes)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

### Routes checkpoint

The reconciliation must show `PASS`.

# Part 6 — Build `bronze_flights`

`flights.csv` is the main fact-like source.

Unlike the three reference files, the source already contains traceability fields:

- `source_record_key`
- `source_period`
- `source_original_filename`
- `source_row_number`

These are **source business/provenance fields and must be preserved exactly** in Bronze.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flights_source
(
  source_record_key STRING,
  flight_date STRING,
  reporting_carrier STRING,
  flight_number STRING,
  tail_number STRING,
  origin_airport_code STRING,
  destination_airport_code STRING,
  scheduled_departure_hhmm STRING,
  actual_departure_hhmm STRING,
  departure_delay_signed_minutes STRING,
  departure_delay_minutes STRING,
  scheduled_arrival_hhmm STRING,
  actual_arrival_hhmm STRING,
  arrival_delay_signed_minutes STRING,
  arrival_delay_minutes STRING,
  cancelled_flag STRING,
  cancellation_code STRING,
  diverted_flag STRING,
  scheduled_elapsed_minutes STRING,
  actual_elapsed_minutes STRING,
  air_time_minutes STRING,
  taxi_out_minutes STRING,
  taxi_in_minutes STRING,
  distance_miles STRING,
  carrier_delay_minutes STRING,
  weather_delay_minutes STRING,
  nas_delay_minutes STRING,
  security_delay_minutes STRING,
  late_aircraft_delay_minutes STRING,
  source_period STRING,
  source_original_filename STRING,
  source_row_number STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/p07-skyops/default/skyops/flights.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);

In [ ]:
%sql
SELECT *
FROM flights_source
LIMIT 10;

In [ ]:
%sql
DESCRIBE flights_source;

In [ ]:
%sql
SELECT COUNT(*) AS flights_source_count
FROM flights_source;

### Flights checkpoint

Confirm that all 31 approved business columns are present and that the source count is obtained from the executed query.

Do not fix invalid HHMM values, delay values, cancellation values or reference relationships here. Those are later data-quality/Silver concerns.

## Add flights Bronze metadata

The following are added by ingestion:

- `_source_file_name`
- `_source_file_path`
- `_ingested_at`
- `_ingestion_run_id`
- `_schema_version`
- `_record_hash`
- `_rescued_payload`

The source's own `source_row_number` remains unchanged.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flights_bronze_ready AS
SELECT
  s.source_record_key,
  s.flight_date,
  s.reporting_carrier,
  s.flight_number,
  s.tail_number,
  s.origin_airport_code,
  s.destination_airport_code,
  s.scheduled_departure_hhmm,
  s.actual_departure_hhmm,
  s.departure_delay_signed_minutes,
  s.departure_delay_minutes,
  s.scheduled_arrival_hhmm,
  s.actual_arrival_hhmm,
  s.arrival_delay_signed_minutes,
  s.arrival_delay_minutes,
  s.cancelled_flag,
  s.cancellation_code,
  s.diverted_flag,
  s.scheduled_elapsed_minutes,
  s.actual_elapsed_minutes,
  s.air_time_minutes,
  s.taxi_out_minutes,
  s.taxi_in_minutes,
  s.distance_miles,
  s.carrier_delay_minutes,
  s.weather_delay_minutes,
  s.nas_delay_minutes,
  s.security_delay_minutes,
  s.late_aircraft_delay_minutes,
  s.source_period,
  s.source_original_filename,
  s.source_row_number,
  'flights.csv' AS _source_file_name,
  '/Volumes/p07-skyops/default/skyops/flights.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version,
  sha2(
    concat_ws(
      '||',
      coalesce(s.source_record_key, '<NULL>'),
      coalesce(s.flight_date, '<NULL>'),
      coalesce(s.reporting_carrier, '<NULL>'),
      coalesce(s.flight_number, '<NULL>'),
      coalesce(s.tail_number, '<NULL>'),
      coalesce(s.origin_airport_code, '<NULL>'),
      coalesce(s.destination_airport_code, '<NULL>'),
      coalesce(s.scheduled_departure_hhmm, '<NULL>'),
      coalesce(s.actual_departure_hhmm, '<NULL>'),
      coalesce(s.departure_delay_signed_minutes, '<NULL>'),
      coalesce(s.departure_delay_minutes, '<NULL>'),
      coalesce(s.scheduled_arrival_hhmm, '<NULL>'),
      coalesce(s.actual_arrival_hhmm, '<NULL>'),
      coalesce(s.arrival_delay_signed_minutes, '<NULL>'),
      coalesce(s.arrival_delay_minutes, '<NULL>'),
      coalesce(s.cancelled_flag, '<NULL>'),
      coalesce(s.cancellation_code, '<NULL>'),
      coalesce(s.diverted_flag, '<NULL>'),
      coalesce(s.scheduled_elapsed_minutes, '<NULL>'),
      coalesce(s.actual_elapsed_minutes, '<NULL>'),
      coalesce(s.air_time_minutes, '<NULL>'),
      coalesce(s.taxi_out_minutes, '<NULL>'),
      coalesce(s.taxi_in_minutes, '<NULL>'),
      coalesce(s.distance_miles, '<NULL>'),
      coalesce(s.carrier_delay_minutes, '<NULL>'),
      coalesce(s.weather_delay_minutes, '<NULL>'),
      coalesce(s.nas_delay_minutes, '<NULL>'),
      coalesce(s.security_delay_minutes, '<NULL>'),
      coalesce(s.late_aircraft_delay_minutes, '<NULL>'),
      coalesce(s.source_period, '<NULL>'),
      coalesce(s.source_original_filename, '<NULL>'),
      coalesce(s.source_row_number, '<NULL>')
    ),
    256
  ) AS _record_hash,
  s._corrupt_record AS _rescued_payload
FROM flights_source s
CROSS JOIN week4_run_control r;

In [ ]:
%sql
SELECT
  source_record_key,
  flight_date,
  reporting_carrier,
  scheduled_departure_hhmm,
  actual_departure_hhmm,
  source_period,
  source_original_filename,
  source_row_number,
  _source_file_name,
  _ingestion_run_id,
  _record_hash,
  _rescued_payload
FROM flights_bronze_ready
LIMIT 10;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_flights
USING DELTA
AS
SELECT *
FROM flights_bronze_ready;

In [ ]:
%sql
DESCRIBE TABLE bronze_flights;

In [ ]:
%sql
DESCRIBE DETAIL bronze_flights;

In [ ]:
%sql
SELECT *
FROM bronze_flights
LIMIT 10;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM flights_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_flights) AS bronze_count,
  (SELECT COUNT(*) FROM bronze_flights)
    - (SELECT COUNT(*) FROM flights_source) AS count_difference,
  CASE
    WHEN (SELECT COUNT(*) FROM flights_source)
       = (SELECT COUNT(*) FROM bronze_flights)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

### Flights checkpoint

The source-to-Bronze reconciliation must show `PASS`.

The Bronze table should contain the 31 source business columns plus the ingestion-level technical metadata.

# Part 7 — Validate the complete Bronze load

All four approved sources should now have persistent Bronze Delta tables.

This section validates the load as one controlled Week 4 batch.

In [ ]:
%sql
SHOW TABLES LIKE 'bronze_*';

### Checkpoint

Confirm that these four tables exist:

```text
bronze_airports
bronze_carriers
bronze_routes
bronze_flights
```

In [ ]:
%sql
WITH counts AS (
  SELECT
    'airports' AS dataset,
    (SELECT COUNT(*) FROM airports_source) AS source_count,
    (SELECT COUNT(*) FROM bronze_airports) AS bronze_count

  UNION ALL

  SELECT
    'carriers',
    (SELECT COUNT(*) FROM carriers_source),
    (SELECT COUNT(*) FROM bronze_carriers)

  UNION ALL

  SELECT
    'routes',
    (SELECT COUNT(*) FROM routes_source),
    (SELECT COUNT(*) FROM bronze_routes)

  UNION ALL

  SELECT
    'flights',
    (SELECT COUNT(*) FROM flights_source),
    (SELECT COUNT(*) FROM bronze_flights)
)
SELECT
  dataset,
  source_count,
  bronze_count,
  bronze_count - source_count AS count_difference,
  CASE
    WHEN source_count = bronze_count THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS status
FROM counts
ORDER BY dataset;

### Exit condition for reconciliation

Every dataset must show:

- `count_difference = 0`
- `status = PASS`

A failure should be investigated before moving to the repeat-run test.

## Validate required technical metadata

Every Bronze row should have the core ingestion metadata populated.

In [ ]:
%sql
WITH metadata_check AS (
  SELECT
    'airports' AS dataset,
    SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END) AS missing_file,
    SUM(CASE WHEN _source_file_path IS NULL THEN 1 ELSE 0 END) AS missing_path,
    SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END) AS missing_time,
    SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END) AS missing_run,
    SUM(CASE WHEN _schema_version IS NULL THEN 1 ELSE 0 END) AS missing_schema,
    SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END) AS missing_hash
  FROM bronze_airports

  UNION ALL

  SELECT
    'carriers',
    SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _source_file_path IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _schema_version IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
  FROM bronze_carriers

  UNION ALL

  SELECT
    'routes',
    SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _source_file_path IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _schema_version IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
  FROM bronze_routes

  UNION ALL

  SELECT
    'flights',
    SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _source_file_path IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _schema_version IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
  FROM bronze_flights
)
SELECT *
FROM metadata_check
ORDER BY dataset;

### Checkpoint

All `missing_*` values should be zero.

A non-zero result means the Bronze-ready view or table write needs investigation.

## Parser/rescued-record context

The `_rescued_payload` column is retained so that parser-related context is not silently discarded.

A non-zero count is **not automatically a Week 4 failure**. It means the source reader captured parser context for some rows; handling such rows belongs to later data-quality work.

In [ ]:
%sql
SELECT
  'airports' AS dataset,
  COUNT(*) AS rows_with_rescued_context
FROM bronze_airports
WHERE _rescued_payload IS NOT NULL

UNION ALL

SELECT
  'carriers',
  COUNT(*)
FROM bronze_carriers
WHERE _rescued_payload IS NOT NULL

UNION ALL

SELECT
  'routes',
  COUNT(*)
FROM bronze_routes
WHERE _rescued_payload IS NOT NULL

UNION ALL

SELECT
  'flights',
  COUNT(*)
FROM bronze_flights
WHERE _rescued_payload IS NOT NULL
ORDER BY dataset;

# Part 8 — Controlled repeat-run test

The Bronze writes use `CREATE OR REPLACE TABLE`.

For the same unchanged source and the same controlled run, rerunning the Bronze writes should refresh the table rather than append a second copy.

### Procedure

1. Record the four counts from the previous reconciliation query.
2. Rerun the four `CREATE OR REPLACE TABLE` cells.
3. Run the count query below.
4. Compare the counts.

Do not change the source files or the run-control values during this test.

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM bronze_airports) AS airports_after_rerun,
  (SELECT COUNT(*) FROM bronze_carriers) AS carriers_after_rerun,
  (SELECT COUNT(*) FROM bronze_routes) AS routes_after_rerun,
  (SELECT COUNT(*) FROM bronze_flights) AS flights_after_rerun;

### Repeat-run checkpoint

The four counts should remain equal to the pre-rerun counts.

If counts increase, inspect whether an append operation was used or whether a different table/write cell was executed.

# Part 9 — Inspect Delta history

Delta history provides evidence that the Bronze tables were created/refreshed and helps explain the controlled rerun.

In [ ]:
%sql
DESCRIBE HISTORY bronze_airports;

In [ ]:
%sql
DESCRIBE HISTORY bronze_carriers;

In [ ]:
%sql
DESCRIBE HISTORY bronze_routes;

In [ ]:
%sql
DESCRIBE HISTORY bronze_flights;

### Checkpoint

Use the history output to explain the table operations performed during Week 4.

Do not claim an exact number of history versions until you inspect your own Databricks result.

# Part 10 — Bronze lineage and source traceability

For a Bronze row, the traceability path should be explainable as:

```text
Source file
   ↓
Source temporary view
   ↓
Bronze-ready view
   ↓
Bronze Delta table
   ↓
Row-level technical metadata
```

For example, `bronze_flights` contains `_source_file_name`, `_source_file_path`, `_ingested_at`, `_ingestion_run_id`, `_schema_version` and `_record_hash`.

Open the Bronze table in Catalog Explorer after the notebook run and inspect its schema, table details and lineage where available.

In [ ]:
%sql
SELECT
  source_record_key,
  _source_file_name,
  _source_file_path,
  _ingested_at,
  _ingestion_run_id,
  _schema_version,
  _record_hash
FROM bronze_flights
LIMIT 10;

### Catalog Explorer evidence

Capture evidence showing:

- the Bronze table exists;
- the table is a Delta table;
- the schema contains the expected source and technical columns;
- lineage is visible if enabled in the workspace.

If lineage is unavailable in the workspace UI, keep the table metadata, source path and notebook SQL as the traceability evidence.

# Part 11 — Week 4 evidence checklist

Capture your own Databricks evidence for:

| Evidence ID | Evidence |
|---|---|
| W04-E01 | Volume listing showing all four approved CSV files |
| W04-E02 | `airports_source` preview/schema/count |
| W04-E03 | `bronze_airports` schema/details/sample |
| W04-E04 | `carriers_source` and `bronze_carriers` evidence |
| W04-E05 | `routes_source` and `bronze_routes` evidence |
| W04-E06 | `flights_source` and `bronze_flights` evidence |
| W04-E07 | Four-source reconciliation showing `PASS` |
| W04-E08 | Metadata completeness result |
| W04-E09 | Repeat-run count comparison |
| W04-E10 | Delta history for the Bronze tables |
| W04-E11 | Catalog Explorer / lineage evidence |

# Part 12 — Week 4 boundary check

Before submitting this notebook, confirm that you have **not** added:

- Silver transformations
- data-quality quarantine tables
- business-rule correction
- deduplication
- HHMM parsing/validation
- route-reference enforcement
- Gold KPIs
- Power BI
- streaming ingestion

Those are later-stage activities in the supplied project plan.

# Week 4 mentor explanation

Be able to explain this sequence without reading the code:

### 1. Why Bronze?

Bronze is the first persistent copy of the approved source. It gives the team a stable, auditable landing layer.

### 2. Why temporary source views?

They give the raw files SQL names so they can be inspected before persistence.

### 3. Why preserve source values?

Because Bronze should remain a faithful representation of the approved source. Cleaning and business interpretation belong later.

### 4. Why add metadata?

The metadata tells us where the row came from, when it was ingested, which controlled run produced it, which schema version was used, and what source content fingerprint it carries.

### 5. Why reconcile counts?

If no filtering or deduplication occurs, every readable source row should reach Bronze. Equal counts provide a basic completeness check.

### 6. Why use `CREATE OR REPLACE TABLE` here?

It provides a controlled full refresh for this fixed teaching batch. Rerunning the same load should not silently append duplicate rows.

### 7. What happens in Week 5?

Silver and data-quality processing can interpret, validate, standardize, quarantine and transform the Bronze data.

# AI Transparency Note

Complete this section before submission:

- **Where AI helped:** 
- **What the team changed after reviewing the AI output:** 
- **What was manually verified in Databricks:** 
- **What every teammate can explain without AI:** 
- **Evidence captured by the team:**

# Week 4 final checklist

- [ ] Four approved source files are visible in the Volume.
- [ ] Correct source views are created.
- [ ] Source schemas were inspected.
- [ ] Source counts were obtained from Databricks.
- [ ] One Bronze Delta table exists for each source.
- [ ] Source business values are preserved.
- [ ] Required technical metadata is populated.
- [ ] Rescued/parser context is retained.
- [ ] Source and Bronze counts reconcile.
- [ ] Repeat-run counts do not unexpectedly increase.
- [ ] Delta history was inspected.
- [ ] Catalog Explorer evidence was captured.
- [ ] No Week 5+ implementation was added.
- [ ] AI Transparency Note is completed.
- [ ] All teammates can explain the Source → Bronze flow.

# 🎉 Week 4 complete

```text
APPROVED SOURCES
      ↓
SOURCE VIEWS
      ↓
BRONZE-READY VIEWS
      ↓
BRONZE DELTA TABLES
      ↓
RECONCILIATION
      ↓
REPEAT-RUN PROOF
      ↓
DELTA HISTORY + CATALOG EVIDENCE
```

**Next stage:** Week 5 — Silver/data-quality processing.

Stop here for Week 4.